In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import os
import copy
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
import torch
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [ ]:
!mkdir -p ~/.kaggle && echo KGAT_7d3d8b9d02289ce21e300dd13afac570 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [ ]:
path = kagglehub.competition_download('dog-breed-identification')

print("Path to competition files:", path)
print(os.listdir(path))

labels = pd.read_csv(f"{path}/labels.csv")
print(labels.head())

train_path = os.path.join(path, "train")
test_path  = os.path.join(path, "test")

In [ ]:
# Собираем пути ко всем тренировочным изображениям
train_files = []
for root, dirs, files in os.walk(train_path):
    for file in files:
        train_files.append(os.path.join(root, file))

print(f"Total train images: {len(train_files)}")

In [ ]:
# Label encoding
encoder = LabelEncoder()
labels['breed_encoded'] = encoder.fit_transform(labels['breed'])
NUM_CLASSES = len(labels['breed'].unique())
print(f"Number of classes: {NUM_CLASSES}")

# маппим id -> label
path_to_label = dict(zip(labels['id'], labels['breed_encoded']))

# Массивы для KFold
train_files = sorted(train_files) # сортим, чтобы на всех фолдах одинаково делились данные
train_ids   = [os.path.splitext(os.path.basename(p))[0] for p in train_files] # берем id
train_lbls  = np.array([path_to_label[i] for i in train_ids]) # берем labels

In [ ]:
# ViT-B/16 обычно использует 224×224
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.12), ratio=(0.3, 3.3))
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
class DogDataset(Dataset):
    def __init__(self, file_paths, label_map, transform=None):
        self.file_paths = file_paths
        self.label_map  = label_map   # dict: img_id -> int
        self.transform  = transform

    def __getitem__(self, i):
        img_path = self.file_paths[i]
        img_id   = os.path.splitext(os.path.basename(img_path))[0]
        img      = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = self.label_map[img_id]
        return img, label

    def __len__(self):
        return len(self.file_paths)


class DogTestDataset(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform  = transform

    def __getitem__(self, i):
        img_path = self.file_paths[i]
        img      = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        img_id = os.path.splitext(os.path.basename(img_path))[0]
        return img, img_id

    def __len__(self):
        return len(self.file_paths)

In [ ]:
def build_model(num_classes):
    model = models.vit_b_16(weights='IMAGENET1K_V1')

    # Замораживаем все слои
    for param in model.parameters():
        param.requires_grad = False

    # Размораживаем голову
    for param in model.heads.parameters():
        param.requires_grad = True

    # Меняем классификатор
    in_features = model.heads.head.in_features
    model.heads.head = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )

    return model


In [ ]:
# Device
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

# Гиперпараметры
N_FOLDS    = 3
EPOCHS     = 15
BATCH_SIZE = 32
LR         = 1e-4
PATIENCE   = 5
MIN_DELTA  = 0.001

# чтобы моделька не сильно уверенной была в выводах
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Пути к тесту
test_files = sorted([
    os.path.join(root, f)
    for root, _, files in os.walk(test_path)
    for f in files
])
test_dataset = DogTestDataset(test_files, transform=val_transform)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# по сути матрица предиктов разных фолдов, по которым потом будем усреднять
oof_test_probs = np.zeros((len(test_files), NUM_CLASSES))

train_files_arr = np.array(train_files)

for fold, (train_idx, val_idx) in enumerate(skf.split(train_files_arr, train_lbls)):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold + 1} / {N_FOLDS}")
    print(f"{'='*60}")

    fold_train_paths = train_files_arr[train_idx].tolist()
    fold_val_paths   = train_files_arr[val_idx].tolist()

    train_dataset = DogDataset(fold_train_paths, path_to_label, transform=train_transform)
    val_dataset   = DogDataset(fold_val_paths,   path_to_label, transform=val_transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)

    model = build_model(NUM_CLASSES).to(device)

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
    )

    best_val_loss     = float('inf')
    patience_counter  = 0
    best_weights      = copy.deepcopy(model.state_dict())

    for epoch in range(EPOCHS):
        # ── Train ──
        model.train()
        total_loss, correct, total = 0, 0, 0

        for images, labels_batch in train_loader:
            images       = images.to(device)
            labels_batch = labels_batch.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            correct    += (outputs.argmax(dim=1) == labels_batch).sum().item()
            total      += labels_batch.size(0)

        train_loss = total_loss / len(train_loader)
        train_acc  = correct / total * 100

        # ── Validation ──
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0

        with torch.no_grad():
            for images, labels_batch in val_loader:
                images       = images.to(device)
                labels_batch = labels_batch.to(device)

                outputs  = model(images)
                loss     = criterion(outputs, labels_batch)

                val_loss    += loss.item()
                val_correct += (outputs.argmax(dim=1) == labels_batch).sum().item()
                val_total   += labels_batch.size(0)

        val_loss = val_loss / len(val_loader)
        val_acc  = val_correct / val_total * 100

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch {epoch+1:>2} | LR: {current_lr:.2e} | "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")

        # Early stopping
        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss    = val_loss
            patience_counter = 0
            best_weights     = copy.deepcopy(model.state_dict())
            print("  ✓ val улучшился, сохранили веса")
        else:
            patience_counter += 1
            print(f"  нет улучшения, patience {patience_counter}/{PATIENCE}")
            if patience_counter >= PATIENCE:
                print(f"  early stopping на эпохе {epoch+1}")
                break

    model.load_state_dict(best_weights)
    print(f"Загрузили лучшие веса фолда {fold+1}, val_loss = {best_val_loss:.4f}")

    model.eval()
    fold_probs = []
    fold_ids   = []

    with torch.no_grad():
        for images, img_ids in test_loader:
            images  = images.to(device)
            outputs = model(images)
            probs   = torch.softmax(outputs, dim=1)
            fold_probs.append(probs.cpu().numpy())
            fold_ids.extend(img_ids)

    fold_probs = np.vstack(fold_probs)
    oof_test_probs += fold_probs / N_FOLDS   # усредняем по фолдам

print("\nОбучение завершено!")

In [ ]:
submission = pd.DataFrame(oof_test_probs, columns=encoder.classes_)
submission.insert(0, "id", fold_ids)

submission.to_csv("submission.csv", index=False)
print(submission.head())
print("submission.csv saved")

In [ ]:
import os
print(os.getcwd())
print(os.path.exists("submission.csv"))
from google.colab import files
files.download("/content/submission.csv")